## 08 — Seleção controlada YOLOv8n-OBB × YOLO11n-OBB (v3)
Este notebook escolhe a família YOLO usando somente treino e validação. Teste
sintético e ASTA não são lidos. Os dois modelos usam o mesmo protocolo, a mesma seed e
os mesmos dados. A recomendação local precisa ser revisada antes de ser publicada.
Execute em runtime GPU. A primeira célula interrompe imediatamente se CUDA não estiver
disponível, antes de montar o Drive ou restaurar o dataset.

In [1]:
import json, platform, sys, torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU obrigatória. No Colab, selecione Ambiente de execução > Alterar tipo de '
        'ambiente > GPU e execute novamente a partir desta célula.'
    )
props=torch.cuda.get_device_properties(0)
livre,total=torch.cuda.mem_get_info(0)
relatorio_gpu={
    'status':'GPU_APROVADA','gpu':torch.cuda.get_device_name(0),
    'cuda':torch.version.cuda,'torch':torch.__version__,
    'vram_total_gb':round(total/1024**3,2),'vram_livre_gb':round(livre/1024**3,2),
    'python':sys.version.split()[0],'plataforma':platform.platform()
}
assert total>=8*1024**3,'GPU com menos de 8 GiB não foi aprovada para este protocolo.'
print(json.dumps(relatorio_gpu,indent=2,ensure_ascii=False))

{
  "status": "GPU_APROVADA",
  "gpu": "Tesla T4",
  "cuda": "12.8",
  "torch": "2.11.0+cu128",
  "vram_total_gb": 14.56,
  "vram_livre_gb": 14.46,
  "python": "3.13.15",
  "plataforma": "Linux-6.6.122+-x86_64-with-glibc2.35"
}


## 1. Ambiente fixado e restauração pelo TAR
O pacote contém somente o dataset aprovado e seus contratos. A restauração transfere
um único arquivo do Drive, verifica SHA-256 e extrai localmente.

In [2]:
%pip install -q ultralytics==8.4.127 opencv-python-headless pandas pyyaml tqdm

from google.colab import drive
from pathlib import Path
import hashlib, json, math, os, shutil, tarfile, tempfile, time
import cv2
import numpy as np
import pandas as pd
import yaml
from tqdm.auto import tqdm

def log_etapa(mensagem):
    print(f"[{time.strftime('%H:%M:%S')}] {mensagem}",flush=True)

def sha256_arquivo(caminho,bloco=8*1024*1024):
    h=hashlib.sha256()
    with open(caminho,'rb') as arq:
        for parte in iter(lambda:arq.read(bloco),b''):h.update(parte)
    return h.hexdigest()

def sha256_com_progresso(caminho,descricao,bloco=8*1024*1024):
    caminho=Path(caminho);total=caminho.stat().st_size;h=hashlib.sha256()
    with caminho.open('rb') as arq,tqdm(total=total,desc=descricao,unit='B',unit_scale=True) as barra:
        for parte in iter(lambda:arq.read(bloco),b''):
            h.update(parte);barra.update(len(parte))
    return h.hexdigest()

def copiar_arquivo_com_progresso(origem,destino,descricao,bloco=8*1024*1024):
    origem,destino=Path(origem),Path(destino);destino.parent.mkdir(parents=True,exist_ok=True)
    total=origem.stat().st_size
    with origem.open('rb') as entrada,destino.open('wb') as saida, \
         tqdm(total=total,desc=descricao,unit='B',unit_scale=True) as barra:
        for parte in iter(lambda:entrada.read(bloco),b''):
            saida.write(parte);barra.update(len(parte))

def copiar_arvore_com_progresso(origem,destino,descricao):
    origem,destino=Path(origem),Path(destino)
    arquivos=sorted(p for p in origem.rglob('*') if p.is_file())
    for arquivo in tqdm(arquivos,desc=descricao,unit='arq'):
        alvo=destino/arquivo.relative_to(origem);alvo.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(arquivo,alvo)

drive.mount('/content/drive',force_remount=True)
BACKUP_DIR=Path('/content/drive/MyDrive/tcc-satellite-streaks')
RUN_ID=(BACKUP_DIR/'data/synthetic_runs/LATEST_RUN_V3.txt').read_text(encoding='utf-8').strip()
PACOTES_DRIVE=BACKUP_DIR/'data/yolo_packages'/RUN_ID
contrato_pacote=json.loads((PACOTES_DRIVE/'contrato_pacote_yolo_v3.json').read_text(encoding='utf-8'))
assert contrato_pacote['run_id']==RUN_ID
pacote_remoto=PACOTES_DRIVE/contrato_pacote['arquivo']
pacote_local=Path('/content')/contrato_pacote['arquivo']

if not pacote_local.exists():
    copiar_arquivo_com_progresso(pacote_remoto,pacote_local,'Restaurando TAR do dataset')
hash_pacote=sha256_com_progresso(pacote_local,'Verificando TAR local')
assert hash_pacote==contrato_pacote['sha256_pacote']

raiz_runs=Path('/content/data/synthetic_runs');raiz_runs.mkdir(parents=True,exist_ok=True)
RUN=raiz_runs/RUN_ID
if RUN.exists():
    assert (RUN/'manifest_geracao_v3.csv').is_file()
    assert sha256_arquivo(RUN/'manifest_geracao_v3.csv')==contrato_pacote['sha256_manifest']
    print('RUN local íntegro reutilizado.')
else:
    temp=Path(tempfile.mkdtemp(prefix='extracao_yolo_',dir='/content'))
    try:
        with tarfile.open(pacote_local,'r') as tar:
            membros=tar.getmembers()
            assert len(membros)==contrato_pacote['membros']
            for membro in membros:
                partes=Path(membro.name).parts
                assert partes and partes[0]==RUN_ID and '..' not in partes
                assert not membro.issym() and not membro.islnk()
            for membro in tqdm(membros,desc='Extraindo dataset local',unit='arq'):
                tar.extract(membro,path=temp,filter='data')
        shutil.move(str(temp/RUN_ID),str(RUN))
    finally:
        if temp.exists():shutil.rmtree(temp)

df=pd.read_csv(RUN/'manifest_geracao_v3.csv')
HASH_MANIFEST=sha256_arquivo(RUN/'manifest_geracao_v3.csv')
validacao=json.loads((RUN/'resumo_validacao_v3.json').read_text(encoding='utf-8'))
assert len(df)==6000 and HASH_MANIFEST==contrato_pacote['sha256_manifest']
assert validacao.get('aprovado_para_treino') is True
for split in ['train','val','test']:
    esperado=set(df.loc[df.split==split,'id'])
    assert {p.stem for p in (RUN/'images'/split).glob('*.png')}==esperado
    assert {p.stem for p in (RUN/'labels'/split).glob('*.txt')}==esperado

dataset_yaml=RUN/'dataset_yolo_selection_v3.yaml'
dataset_yaml.write_text(yaml.safe_dump({
    'path':str(RUN.resolve()),'train':'images/train','val':'images/val','test':'images/test',
    'names':{0:'satellite_streak'}
},sort_keys=False),encoding='utf-8')
print(json.dumps({
    'status':'DATASET_GPU_LOCAL_APROVADO','run_id':RUN_ID,
    'sha256_manifest':HASH_MANIFEST,'amostras':len(df),
    'pacote_sha256':hash_pacote
},indent=2,ensure_ascii=False))

Mounted at /content/drive


Verificando TAR local:   0%|          | 0.00/861M [00:00<?, ?B/s]

RUN local íntegro reutilizado.
{
  "status": "DATASET_GPU_LOCAL_APROVADO",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "amostras": 6000,
  "pacote_sha256": "163f46b536aa416c39c806490d6e6066b393bdb53a1e6de5911f24699a4b5434"
}


## 2. Contrato pré-registrado da seleção
A seleção usa uma seed e somente validação. O vencedor será treinado depois em mais duas
seeds. Teste sintético e ASTA permanecem proibidos nesta etapa.

In [3]:
DRIVE_EXP=BACKUP_DIR/'experiments/yolo_family_selection_v3'/RUN_ID
DRIVE_EXP.mkdir(parents=True,exist_ok=True)
CONFIG_SELECTION={
    'protocolo':'yolo_family_selection_v1','run_id':RUN_ID,
    'sha256_manifest':HASH_MANIFEST,'ultralytics':'8.4.127',
    'modelos':{'yolov8n_obb':'yolov8n-obb.pt','yolo11n_obb':'yolo11n-obb.pt'},
    'seed_selecao':20260823,'seeds_finais':[20260823,20260824,20260825],
    'treino':{
        'epochs':100,'patience':20,'imgsz':512,'batch':16,'optimizer':'AdamW',
        'lr0':0.001,'lrf':0.01,'cos_lr':True,'weight_decay':0.0005,
        'warmup_epochs':3.0,'deterministic':True,'amp':True,'workers':2,
        'mosaic':0.0,'mixup':0.0,'copy_paste':0.0,
        'hsv_h':0.0,'hsv_s':0.0,'hsv_v':0.0,'degrees':0.0,
        'translate':0.0,'scale':0.0,'shear':0.0,'perspective':0.0,
        'flipud':0.0,'fliplr':0.0
    },
    'predicao_val':{'conf_min':0.001,'iou_nms':0.7,'max_det':100},
    'thresholds_confianca':[.01,.02,.03,.05,.075,.10,.15,.20,.25,.30,.35,.40,.45,.50,.55,.60,.65,.70,.75,.80,.85,.90,.95],
    'matching':{'iou':0.30,'angulo':15.0,'distancia':14.0,'cobertura':0.50,'modo':'combinado'},
    'regra_escolha':{
        'delta_f1_material':0.02,'delta_fp_frames_material':0.01,
        'delta_map_material':0.01,'desempate_final':'yolo11n_obb'
    },
    'restricoes':{
        'teste_sintetico_lido':False,'asta_lido':False,
        'modelo_perdedor_nao_sera_avaliado_no_teste':True
    }
}
texto_config=json.dumps(CONFIG_SELECTION,sort_keys=True,separators=(',',':'))
HASH_CONFIG_SELECTION=hashlib.sha256(texto_config.encode()).hexdigest()
CONFIG_SELECTION['sha256_config_selecao']=HASH_CONFIG_SELECTION
config_path=RUN/'protocolo_selecao_yolo_v3.json'
config_path.write_text(json.dumps(CONFIG_SELECTION,indent=2),encoding='utf-8')
remoto=DRIVE_EXP/config_path.name
if remoto.exists():
    existente=json.loads(remoto.read_text(encoding='utf-8'))
    assert existente==CONFIG_SELECTION,'Já existe protocolo remoto divergente; seleção bloqueada.'
else:
    shutil.copy2(config_path,remoto)
print(json.dumps(CONFIG_SELECTION,indent=2,ensure_ascii=False))
print('Contrato de seleção congelado antes do treino.')

{
  "protocolo": "yolo_family_selection_v1",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "ultralytics": "8.4.127",
  "modelos": {
    "yolov8n_obb": "yolov8n-obb.pt",
    "yolo11n_obb": "yolo11n-obb.pt"
  },
  "seed_selecao": 20260823,
  "seeds_finais": [
    20260823,
    20260824,
    20260825
  ],
  "treino": {
    "epochs": 100,
    "patience": 20,
    "imgsz": 512,
    "batch": 16,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "cos_lr": true,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "deterministic": true,
    "amp": true,
    "workers": 2,
    "mosaic": 0.0,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.0,
    "degrees": 0.0,
    "translate": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.0
  },
  "predicao_val": {
    "conf_min": 0.001,
    "iou_nms": 0.7,
    

## 3. Treino comparativo retomável
Cada família é treinada com a mesma configuração. Um checkpoint é copiado ao Drive a
cada cinco épocas. Resultados concluídos e compatíveis são reutilizados.

In [4]:
EXECUTAR_SELECAO=True
if not EXECUTAR_SELECAO:
    print('Treino bloqueado. Confira GPU, dataset e contrato; depois altere a chave para True.')
else:
    import ultralytics
    from ultralytics import YOLO
    assert ultralytics.__version__=='8.4.127',f'Versão inesperada: {ultralytics.__version__}'
    LOCAL_PROJECT=Path('/content/yolo_family_selection_v3')/RUN_ID
    LOCAL_PROJECT.mkdir(parents=True,exist_ok=True)
    registros_treino=[]

    def callback_checkpoint(destino,modelo_chave,hash_peso_inicial):
        def salvar(trainer):
            epoca=int(trainer.epoch)+1
            if epoca%5!=0 and not trainer.stop:return
            destino.mkdir(parents=True,exist_ok=True)
            for caminho in [Path(trainer.last),Path(trainer.best),Path(trainer.save_dir)/'results.csv',Path(trainer.save_dir)/'args.yaml']:
                if caminho.is_file():shutil.copy2(caminho,destino/caminho.name)
            estado={'modelo':modelo_chave,'epoca':epoca,'sha256_config_selecao':HASH_CONFIG_SELECTION,
                    'sha256_peso_inicial':hash_peso_inicial}
            (destino/'estado_checkpoint.json').write_text(json.dumps(estado,indent=2),encoding='utf-8')
        return salvar

    for modelo_chave,checkpoint in CONFIG_SELECTION['modelos'].items():
        remoto_modelo=DRIVE_EXP/'treinos'/modelo_chave
        conclusao_remota=remoto_modelo/'conclusao_treino.json'
        if conclusao_remota.exists():
            registro=json.loads(conclusao_remota.read_text(encoding='utf-8'))
            assert registro['sha256_config_selecao']==HASH_CONFIG_SELECTION
            assert (remoto_modelo/'weights/best.pt').is_file()
            registros_treino.append(registro)
            print(modelo_chave,'já concluído e reutilizado.')
            continue

        modelo_inicial=YOLO(checkpoint)
        peso_inicial=Path(checkpoint)
        assert peso_inicial.is_file(),f'Peso inicial não encontrado: {checkpoint}'
        hash_peso_inicial=sha256_arquivo(peso_inicial)
        checkpoint_remoto=remoto_modelo/'checkpoint'
        ultimo_remoto=checkpoint_remoto/'last.pt'
        nome_execucao=f'{modelo_chave}_seed_{CONFIG_SELECTION["seed_selecao"]}'
        destino_local=LOCAL_PROJECT/nome_execucao

        if ultimo_remoto.exists():
            pesos_local=destino_local/'weights';pesos_local.mkdir(parents=True,exist_ok=True)
            for nome in ['last.pt','best.pt']:
                if (checkpoint_remoto/nome).exists():shutil.copy2(checkpoint_remoto/nome,pesos_local/nome)
            for nome in ['results.csv','args.yaml']:
                if (checkpoint_remoto/nome).exists():shutil.copy2(checkpoint_remoto/nome,destino_local/nome)
            modelo=YOLO(pesos_local/'last.pt')
            modelo.add_callback('on_model_save',callback_checkpoint(checkpoint_remoto,modelo_chave,hash_peso_inicial))
            print('Retomando',modelo_chave,'do checkpoint remoto.')
            modelo.train(resume=True,device=0,workers=CONFIG_SELECTION['treino']['workers'])
        else:
            modelo=modelo_inicial
            modelo.add_callback('on_model_save',callback_checkpoint(checkpoint_remoto,modelo_chave,hash_peso_inicial))
            args={**CONFIG_SELECTION['treino'],'data':str(dataset_yaml),'seed':CONFIG_SELECTION['seed_selecao'],
                  'device':0,'project':str(LOCAL_PROJECT),'name':nome_execucao,'exist_ok':True,
                  'pretrained':True,'val':True,'plots':True,'save':True,'save_period':5,
                  'cache':False,'verbose':True}
            modelo.train(**args)

        pasta_resultado=Path(modelo.trainer.save_dir)
        best=pasta_resultado/'weights/best.pt';last=pasta_resultado/'weights/last.pt'
        assert best.is_file() and last.is_file() and (pasta_resultado/'results.csv').is_file()
        copiar_arvore_com_progresso(pasta_resultado,remoto_modelo,f'Backup final {modelo_chave}')
        registro={
            'modelo':modelo_chave,'checkpoint_inicial':checkpoint,
            'sha256_peso_inicial':hash_peso_inicial,'seed':CONFIG_SELECTION['seed_selecao'],
            'sha256_config_selecao':HASH_CONFIG_SELECTION,
            'sha256_best_pt':sha256_arquivo(best),'sha256_last_pt':sha256_arquivo(last),
            'epocas_registradas':int(len(pd.read_csv(pasta_resultado/'results.csv'))),
            'concluido':True
        }
        (remoto_modelo/'conclusao_treino.json').write_text(json.dumps(registro,indent=2),encoding='utf-8')
        registros_treino.append(registro)
        print(json.dumps(registro,indent=2,ensure_ascii=False))
    print('Treinos de seleção concluídos:',[r['modelo'] for r in registros_treino])

yolov8n_obb já concluído e reutilizado.
yolo11n_obb já concluído e reutilizado.
Treinos de seleção concluídos: ['yolov8n_obb', 'yolo11n_obb']


In [5]:
for modelo in ['yolov8n_obb', 'yolo11n_obb']:
    caminho = DRIVE_EXP / 'treinos' / modelo / 'conclusao_treino.json'

    assert caminho.is_file(), f'Registro ausente: {modelo}'

    registro = json.loads(caminho.read_text(encoding='utf-8'))

    print('\n' + '=' * 70)
    print(modelo)
    print(json.dumps(registro, indent=2, ensure_ascii=False))


yolov8n_obb
{
  "modelo": "yolov8n_obb",
  "checkpoint_inicial": "yolov8n-obb.pt",
  "sha256_peso_inicial": "fa6e4cd2691f132875c143135affaa66b5d89394ebb1d07d19770a9b6382c1b8",
  "seed": 20260823,
  "sha256_config_selecao": "25af65bbb44f0b2d84c7064ecebe698dc81e74df842b85e1bcb2426d439ff0b1",
  "sha256_best_pt": "f6fc2ce471437ede9663d45bc9ba21d4c65dcde8e59244d3491783fb4796a09a",
  "sha256_last_pt": "860a1d737566963128136d7a97e38b75a5956f825fab211dbe0e486feb4da7d0",
  "epocas_registradas": 100,
  "concluido": true
}

yolo11n_obb
{
  "modelo": "yolo11n_obb",
  "checkpoint_inicial": "yolo11n-obb.pt",
  "sha256_peso_inicial": "b62898ebf38940ca4df323863e45ee9d84a1a46d5d11ebdde529fb33aa9f3a32",
  "seed": 20260823,
  "sha256_config_selecao": "25af65bbb44f0b2d84c7064ecebe698dc81e74df842b85e1bcb2426d439ff0b1",
  "sha256_best_pt": "dab6ececdc1011a645bfa4a3486ba3bd10746d083fa3ccc61ac462fc21b1aa7e",
  "sha256_last_pt": "a0d6bb6c3a7bf5b48991834e8f0cfe2e6f3bd1bf286d1a211f5172ac55f5ff21",
  "epocas_reg

## 4. Avaliação comum na validação
O confidence threshold é calibrado separadamente para cada família na mesma grade. O
matching combinado usa exatamente os limiares geométricos congelados no baseline Hough.

In [6]:
EXECUTAR_AVALIACAO_VALIDACAO=True
if not EXECUTAR_AVALIACAO_VALIDACAO:
    print('Avaliação bloqueada. Execute somente depois dos dois treinos concluídos.')
else:
    from ultralytics import YOLO
    import gc,torch

    # Uma execução interrompida pode deixar o gerador/modelo anterior referenciado.
    for nome_residual in ['resultados','resultado','modelo','metricas','caixa']:
        globals().pop(nome_residual,None)
    gc.collect();torch.cuda.empty_cache()

    def parametros_linha(p0,p1):
        p0,p1=np.asarray(p0,dtype=float),np.asarray(p1,dtype=float)
        vetor=p1-p0;comp=float(np.linalg.norm(vetor));direcao=vetor/comp if comp>0 else np.array([1.,0.])
        return (p0+p1)/2,direcao,comp

    def erro_angular(d1,d2):
        return float(np.degrees(np.arccos(np.clip(abs(np.dot(d1,d2)),-1,1))))

    def distancia_ponto_linha(ponto,centro,direcao):
        v=np.asarray(ponto)-np.asarray(centro)
        return float(np.linalg.norm(v-np.dot(v,direcao)*direcao))

    def cobertura_projetada(p0a,p1a,p0b,p1b):
        cb,db,lb=parametros_linha(p0b,p1b)
        if lb<=0:return 0.0
        t=sorted([np.dot(np.asarray(p0a)-cb,db),np.dot(np.asarray(p1a)-cb,db)])
        inter=max(0,min(t[1],lb/2)-max(t[0],-lb/2))
        return float(inter/lb)

    def centerline(poligono):
        p=np.asarray(poligono,dtype=float)
        d01=np.linalg.norm(p[0]-p[1]);d12=np.linalg.norm(p[1]-p[2])
        if d01>=d12:return (p[0]+p[3])/2,(p[1]+p[2])/2
        return (p[0]+p[1])/2,(p[2]+p[3])/2

    def iou_poligonos(a,b):
        a,b=np.asarray(a,np.float32),np.asarray(b,np.float32)
        area_a=abs(float(cv2.contourArea(a)));area_b=abs(float(cv2.contourArea(b)))
        inter=float(cv2.intersectConvexConvex(a,b)[0]);uniao=area_a+area_b-inter
        return inter/uniao if uniao>0 else 0.0

    def ler_gt(linha):
        tokens=(RUN/'labels'/linha.split/(linha.id+'.txt')).read_text(encoding='utf-8').split()
        if not tokens:return None
        valores=np.asarray([float(x) for x in tokens[1:9]]).reshape(4,2)
        return valores*np.array([512.,512.])

    def metrica_predicao(pred,gt):
        p0,p1=centerline(pred);g0,g1=centerline(gt)
        cp,dp,_=parametros_linha(p0,p1);cg,dg,_=parametros_linha(g0,g1)
        return {'iou':iou_poligonos(pred,gt),'angulo':erro_angular(dp,dg),
                'distancia':distancia_ponto_linha(cp,cg,dg),
                'cobertura':cobertura_projetada(p0,p1,g0,g1)}

    def escolher_match(mets,lim):
        candidatos=[]
        for indice,m in enumerate(mets):
            miou=m['iou']>=lim['iou']
            mgeo=m['angulo']<=lim['angulo'] and m['distancia']<=lim['distancia'] and m['cobertura']>=lim['cobertura']
            if not (miou or mgeo):continue
            si=m['iou']/lim['iou'] if miou else -np.inf
            sg=((1-m['angulo']/lim['angulo'])+(1-m['distancia']/lim['distancia'])+m['cobertura']) if mgeo else -np.inf
            candidatos.append((max(si,sg),indice,m))
        return max(candidatos,key=lambda x:x[0]) if candidatos else None

    def avaliar_threshold(dados,predicoes,threshold):
        saida=[];lim=CONFIG_SELECTION['matching']
        for linha in dados.itertuples():
            grupo=predicoes[(predicoes.id==linha.id)&(predicoes.conf>=threshold)]
            polys=[g[['x1','y1','x2','y2','x3','y3','x4','y4']].to_numpy(float).reshape(4,2) for _,g in grupo.iterrows()]
            gt=ler_gt(linha)
            base={'id':linha.id,'tipo':linha.tipo,'subtipo':linha.subtipo,
                  'sha256_fundo':linha.sha256_fundo,'n_predicoes':len(polys)}
            if gt is None:
                base.update({'tp':0,'fp':len(polys),'fn':0,'acertou':False})
            else:
                mets=[metrica_predicao(p,gt) for p in polys];match=escolher_match(mets,lim)
                base.update({'tp':int(match is not None),'fp':max(0,len(polys)-int(match is not None)),
                             'fn':int(match is None),'acertou':match is not None})
            saida.append(base)
        resultado=pd.DataFrame(saida);tp=int(resultado.tp.sum());fp=int(resultado.fp.sum());fn=int(resultado.fn.sum())
        p=tp/(tp+fp) if tp+fp else 0;r=tp/(tp+fn) if tp+fn else 0;f1=2*p*r/(p+r) if p+r else 0
        neg=resultado[resultado.tipo=='negativo']
        resumo={'threshold':float(threshold),'TP':tp,'FP':fp,'FN':fn,'precisao':p,'recall':r,'f1':f1,
                'fp_por_imagem_negativa':float(neg.fp.mean()),
                'taxa_frames_negativos_com_fp':float((neg.fp>0).mean())}
        return resultado,resumo

    def escolher_familia(tabela,regra):
        assert set(tabela.modelo)=={'yolov8n_obb','yolo11n_obb'} and len(tabela)==2
        a,b=tabela.sort_values('modelo').iloc[0],tabela.sort_values('modelo').iloc[1]
        if abs(a.f1-b.f1)>=regra['delta_f1_material']:
            return (a if a.f1>b.f1 else b),'maior F1 combinado com diferença material'
        if abs(a.taxa_frames_negativos_com_fp-b.taxa_frames_negativos_com_fp)>=regra['delta_fp_frames_material']:
            return (a if a.taxa_frames_negativos_com_fp<b.taxa_frames_negativos_com_fp else b),'menor taxa de frames negativos com FP'
        if abs(a.map50_95_obb-b.map50_95_obb)>=regra['delta_map_material']:
            return (a if a.map50_95_obb>b.map50_95_obb else b),'maior mAP50-95 OBB'
        return tabela[tabela.modelo==regra['desempate_final']].iloc[0],'desempate pré-registrado por maturidade/eficiência'

    df_val=df[df.split=='val'].copy().sort_values('id')
    ids_val_esperados=set(df_val.id.astype(str))
    pasta_imagens_val=RUN/'images/val'
    assert pasta_imagens_val.is_dir()
    assert {p.stem for p in pasta_imagens_val.glob('*.png')}==ids_val_esperados
    candidatos=[];ARTEFATOS_AVALIACAO=RUN/'yolo_selection_validation';ARTEFATOS_AVALIACAO.mkdir(exist_ok=True)
    for modelo_chave in CONFIG_SELECTION['modelos']:
        gc.collect();torch.cuda.empty_cache()
        remoto_modelo=DRIVE_EXP/'treinos'/modelo_chave
        conclusao=json.loads((remoto_modelo/'conclusao_treino.json').read_text(encoding='utf-8'))
        best_remoto=remoto_modelo/'weights/best.pt'
        assert sha256_arquivo(best_remoto)==conclusao['sha256_best_pt']
        modelo=YOLO(best_remoto)
        registros=[];tempos=[];ids_val_observados=[]
        # Passar uma lista de 900 caminhos pode virar um lote único no Ultralytics,
        # mesmo com stream=True. Um diretório + batch=1 mantém uso de VRAM limitado.
        resultados=modelo.predict(source=str(pasta_imagens_val),stream=True,batch=1,imgsz=CONFIG_SELECTION['treino']['imgsz'],
            conf=CONFIG_SELECTION['predicao_val']['conf_min'],iou=CONFIG_SELECTION['predicao_val']['iou_nms'],
            max_det=CONFIG_SELECTION['predicao_val']['max_det'],device=0,verbose=False)
        for resultado in tqdm(resultados,total=len(ids_val_esperados),desc=f'Predições {modelo_chave}',unit='img'):
            id_amostra=Path(resultado.path).stem;ids_val_observados.append(id_amostra)
            tempos.append(float(resultado.speed.get('inference',np.nan)))
            if resultado.obb is None:continue
            polys=resultado.obb.xyxyxyxy.cpu().numpy();confs=resultado.obb.conf.cpu().numpy()
            for poly,conf in zip(polys,confs):
                plano=poly.reshape(-1)
                registros.append({'id':id_amostra,'conf':float(conf),
                                  **{f'{eixo}{i+1}':float(plano[2*i+(0 if eixo=="x" else 1)]) for i in range(4) for eixo in ['x','y']}})
        assert len(ids_val_observados)==len(ids_val_esperados)
        assert len(set(ids_val_observados))==len(ids_val_observados)
        assert set(ids_val_observados)==ids_val_esperados
        del resultados,resultado
        gc.collect();torch.cuda.empty_cache()
        pred=pd.DataFrame(registros,columns=['id','conf','x1','y1','x2','y2','x3','y3','x4','y4'])
        pred.to_csv(ARTEFATOS_AVALIACAO/f'predicoes_val_{modelo_chave}.csv',index=False)
        grades=[];resultados_por_threshold={}
        for threshold in CONFIG_SELECTION['thresholds_confianca']:
            por_imagem,resumo=avaliar_threshold(df_val,pred,threshold)
            grades.append(resumo);resultados_por_threshold[threshold]=por_imagem
        grade=pd.DataFrame(grades).sort_values(['f1','precisao','threshold'],ascending=[False,False,False])
        melhor=grade.iloc[0].to_dict();threshold=float(melhor['threshold'])
        grade.to_csv(ARTEFATOS_AVALIACAO/f'grade_threshold_val_{modelo_chave}.csv',index=False)
        resultados_por_threshold[threshold].to_csv(ARTEFATOS_AVALIACAO/f'resultado_val_{modelo_chave}.csv',index=False)
        metricas=modelo.val(data=str(dataset_yaml),split='val',imgsz=CONFIG_SELECTION['treino']['imgsz'],
                            batch=CONFIG_SELECTION['treino']['batch'],device=0,workers=2,plots=False,verbose=False,
                            project=str(ARTEFATOS_AVALIACAO/'ultralytics'),name=modelo_chave,exist_ok=True)
        caixa=metricas.box
        candidato={'modelo':modelo_chave,**melhor,'map50_95_obb':float(caixa.map),'map50_obb':float(caixa.map50),
                   'inferencia_ms_media':float(np.nanmean(tempos)),'sha256_best_pt':conclusao['sha256_best_pt']}
        candidatos.append(candidato)
        del metricas,caixa,modelo
        gc.collect();torch.cuda.empty_cache()

    tabela=pd.DataFrame(candidatos).sort_values('modelo').reset_index(drop=True)
    regra=CONFIG_SELECTION['regra_escolha']
    vencedor,motivo=escolher_familia(tabela,regra)
    comparacao_path=ARTEFATOS_AVALIACAO/'comparacao_modelos_val_v3.csv'
    tabela.to_csv(comparacao_path,index=False)
    recomendacao={'protocolo':'yolo_family_selection_v1','run_id':RUN_ID,
                  'sha256_manifest':HASH_MANIFEST,'sha256_config_selecao':HASH_CONFIG_SELECTION,
                  'vencedor_recomendado':vencedor.modelo,'motivo':motivo,
                  'threshold_confianca':float(vencedor.threshold),
                  'sha256_best_pt':vencedor.sha256_best_pt,
                  'sha256_tabela_comparacao':sha256_arquivo(comparacao_path),
                  'teste_sintetico_lido':False,'asta_lido':False}
    (ARTEFATOS_AVALIACAO/'recomendacao_local_v3.json').write_text(json.dumps(recomendacao,indent=2),encoding='utf-8')
    copiar_arvore_com_progresso(ARTEFATOS_AVALIACAO,DRIVE_EXP/'avaliacao_val','Backup avaliação de validação')
    print(tabela[['modelo','threshold','precisao','recall','f1','taxa_frames_negativos_com_fp','map50_95_obb','inferencia_ms_media']])
    print(json.dumps(recomendacao,indent=2,ensure_ascii=False))
    print('Recomendação local calculada. Publicação do vencedor ainda bloqueada.')

Predições yolov8n_obb:   0%|          | 0/900 [00:00<?, ?img/s]

Ultralytics 8.4.127 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2635.3±1586.8 MB/s, size: 129.3 KB)
val: Scanning /content/data/synthetic_runs/synthetic_v3_c41d5091de56/labels/val... 900 images, 450 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 900/900 1.7Kit/s 0.5s
val: New cache created: /content/data/synthetic_runs/synthetic_v3_c41d5091de56/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 5.9it/s 9.6s
                   all        900        450      0.925      0.793      0.893      0.559
Speed: 0.8ms preprocess, 3.0ms inference, 0.0ms loss, 3.9ms postprocess per image


Predições yolo11n_obb:   0%|          | 0/900 [00:00<?, ?img/s]

Ultralytics 8.4.127 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3514.4±914.1 MB/s, size: 143.1 KB)
val: Scanning /content/data/synthetic_runs/synthetic_v3_c41d5091de56/labels/val.cache... 900 images, 450 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 900/900 343.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 6.4it/s 8.9s
                   all        900        450      0.905      0.958      0.977      0.551
Speed: 1.1ms preprocess, 3.0ms inference, 0.0ms loss, 2.7ms postprocess per image


Backup avaliação de validação:   0%|          | 0/8 [00:00<?, ?arq/s]

        modelo  threshold  precisao    recall        f1  \
0  yolo11n_obb       0.45  0.940678  0.986667  0.963124   
1  yolov8n_obb       0.45  0.961290  0.993333  0.977049   

   taxa_frames_negativos_com_fp  map50_95_obb  inferencia_ms_media  
0                      0.002222      0.550573             9.473010  
1                      0.000000      0.559282             7.212659  
{
  "protocolo": "yolo_family_selection_v1",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "sha256_config_selecao": "25af65bbb44f0b2d84c7064ecebe698dc81e74df842b85e1bcb2426d439ff0b1",
  "vencedor_recomendado": "yolo11n_obb",
  "motivo": "desempate pré-registrado por maturidade/eficiência",
  "threshold_confianca": 0.45,
  "sha256_best_pt": "dab6ececdc1011a645bfa4a3486ba3bd10746d083fa3ccc61ac462fc21b1aa7e",
  "sha256_tabela_comparacao": "6c771ed990644e296b07991088c5d0eaf0f06316f3376fd9763f83598ba26827",
  "teste_sintetico_li

## 5. Gate humano e publicação do vencedor
Só altere a chave depois de revisar a tabela completa. A publicação não toca no teste e
apenas transforma a recomendação determinística em contrato oficial de seleção.

In [7]:
APROVAR_SELECAO_YOLO=True
recomendacao_path=RUN/'yolo_selection_validation/recomendacao_local_v3.json'
if not APROVAR_SELECAO_YOLO:
    print('Seleção não publicada. Compartilhe a tabela e a recomendação para revisão.')
else:
    recomendacao=json.loads(recomendacao_path.read_text(encoding='utf-8'))
    assert recomendacao['sha256_manifest']==HASH_MANIFEST
    assert recomendacao['sha256_config_selecao']==HASH_CONFIG_SELECTION
    comparacao_path=RUN/'yolo_selection_validation/comparacao_modelos_val_v3.csv'
    assert sha256_arquivo(comparacao_path)==recomendacao['sha256_tabela_comparacao']
    best_remoto=DRIVE_EXP/'treinos'/recomendacao['vencedor_recomendado']/'weights/best.pt'
    assert sha256_arquivo(best_remoto)==recomendacao['sha256_best_pt']
    contrato={**recomendacao,'aprovado':True,'seeds_finais':CONFIG_SELECTION['seeds_finais'],
              'regra_escolha':CONFIG_SELECTION['regra_escolha']}
    texto=json.dumps(contrato,sort_keys=True,separators=(',',':'))
    contrato['sha256_contrato_selecao']=hashlib.sha256(texto.encode()).hexdigest()
    contrato_path=DRIVE_EXP/'contrato_selecao_yolo_v3.json'
    if contrato_path.exists():
        assert json.loads(contrato_path.read_text(encoding='utf-8'))==contrato,'Contrato remoto divergente.'
    else:
        contrato_path.write_text(json.dumps(contrato,indent=2),encoding='utf-8')
    ponteiro=BACKUP_DIR/'experiments/yolo_family_selection_v3/LATEST_SELECTION_V3.json'
    ponteiro.parent.mkdir(parents=True,exist_ok=True)
    ponteiro.write_text(json.dumps({
        'run_id':RUN_ID,'sha256_manifest':HASH_MANIFEST,
        'sha256_contrato_selecao':contrato['sha256_contrato_selecao'],
        'vencedor':contrato['vencedor_recomendado']
    },indent=2),encoding='utf-8')
    print(json.dumps(contrato,indent=2,ensure_ascii=False))
    print('SELEÇÃO YOLO PUBLICADA. Teste sintético e ASTA permanecem bloqueados.')

{
  "protocolo": "yolo_family_selection_v1",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "sha256_config_selecao": "25af65bbb44f0b2d84c7064ecebe698dc81e74df842b85e1bcb2426d439ff0b1",
  "vencedor_recomendado": "yolo11n_obb",
  "motivo": "desempate pré-registrado por maturidade/eficiência",
  "threshold_confianca": 0.45,
  "sha256_best_pt": "dab6ececdc1011a645bfa4a3486ba3bd10746d083fa3ccc61ac462fc21b1aa7e",
  "sha256_tabela_comparacao": "6c771ed990644e296b07991088c5d0eaf0f06316f3376fd9763f83598ba26827",
  "teste_sintetico_lido": false,
  "asta_lido": false,
  "aprovado": true,
  "seeds_finais": [
    20260823,
    20260824,
    20260825
  ],
  "regra_escolha": {
    "delta_f1_material": 0.02,
    "delta_fp_frames_material": 0.01,
    "delta_map_material": 0.01,
    "desempate_final": "yolo11n_obb"
  },
  "sha256_contrato_selecao": "bc89537e86230ce1757ae990054859b59c59bbfcdfae8ef3daf0a4f13f5cd391"
}
SEL